# WaveSeekerNet Demonstration with Real IAV Data

This notebook demonstrates how to use the `waveseekernet` package for predicting Influenza A virus subtypes or host sources. 

We will:
1. Load data.
2. Initialize the `WaveSeekerClassifier`.
3. Perform a short training run.
4. Evaluate the model performance.

In [1]:
import numpy as np
import torch
from WaveSeekerNet import WaveSeekerClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score
import shap
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import balanced_accuracy_score as ba_score
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, matthews_corrcoef
import random
from sampling import get_rare_sequence, resampling
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/hnguyen/miniconda3/envs/WaveSeekerNet-env/lib/python3.12/site-packages/pytorch_wavelets/dtcwt/coeffs.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
/home/hnguyen/miniconda3/envs/WaveSeekerNet-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.4.1+cu121
CUDA available: True


In [2]:
def set_seed(random_seed):
    print ("Set Global Seed\n")
    torch.manual_seed(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)

In [3]:
def get_score(y_true, y_pred):
    ba = ba_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    p_score = precision_score(y_true, y_pred, average="macro")
    r_score = recall_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)
    print(ba, f1, p_score, r_score, mcc)
    print(classification_report(y_true, y_pred, zero_division=np.nan))
    return ba, f1, p_score, r_score, mcc

In [4]:
train_path      = '/home/hnguyen/Documents/PhD/Part3/Data/numpy_data/01_PB2/'
test_path       = '/home/hnguyen/Documents/PhD/Part3/Data/numpy_data/01_PB2/' 

In [5]:
X_train = np.load(train_path + 'X_train_onehot.npy')
y_train = np.load(train_path + 'y_train.npy')

In [6]:
X_test_high_quality = np.load(test_path + 'X_test_onehot.npy')
y_test_high_quality = np.load(test_path + 'y_test.npy')

In [7]:
print("Train data shape:", X_train.shape, y_train.shape)
print("Test High-quality Data Shape:", X_test_high_quality.shape, y_test_high_quality.shape)

Train data shape: (78732, 5, 2400) (78732,)
Test High-quality Data Shape: (83325, 5, 2400) (83325,)


In [8]:
n_out = len(np.unique(y_train))
seq_len = X_train.shape[2]
res_len = X_train.shape[1]
patch_size = (12, res_len)
epochs = 35
batch_size = 256
emb_dim = 64
final_hidden_size = 24
n_splits = 10

In [9]:
params_dict = {"use_fft": False,  # default True
               "use_wavelets": False,  # default True
               "use_gmlp": False,
               "activation_mish": torch.nn.Mish,  # default ErMish
               "activation_gelu": torch.nn.GELU,
               "activation_relu": torch.nn.ReLU,
               "use_kan": False,  # default True
               "use_smoe": False,  # default True
               "use_gc": False,  # default True
               "use_lookahead": False,  # default True
               }

In [10]:
cv_cols = ["Model", "Balanced Accuracy", "F1-Score (Macro)", "Precision (Macro)", "Recall (Macro)", "MCC"]
param_results_high_quality = []

In [11]:
splitter = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=1, random_state=0)

In [12]:
seed = 0
set_seed(seed) # old is 0

Set Global Seed



In [13]:
clf_load_weight = WaveSeekerClassifier(
    n_channels=1,
    seq_L=seq_len,
    res_L=res_len,
    patch_size=patch_size,
    n_out=n_out,
    batch_size=batch_size,
    emb_dim=emb_dim,
    final_hidden_size=final_hidden_size,
    epochs=epochs,
    patch_mode="patch",
    wavelet_names=["sym4"],
    n_blocks=1,
    lr=0.0025)

In [14]:
# clf_load_weight.load_weights("/home/hnguyen/Documents/PhD/Part3/Data/model_weights/01_PB2/Ablation_weight_0_Baseline.pt")

In [15]:
clf_load_weight.load_weights("/home/hnguyen/Documents/PhD/Part3/Data/model_weights/01_PB2/Ablation_weight_0_Baseline.pt")

Initializing model...
INFO | WaveSeekerNet.model | Using device: cuda:0


In [16]:
_, logits = clf_load_weight.predict(X_test_high_quality, return_logits=True)

In [17]:
ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, np.argmax(logits, axis=1))

0.9544679340073686 0.9240603733656879 0.8980866613253534 0.9544679340073686 0.9421375140541737
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     73749
           1       0.96      1.00      0.98      7495
           2       0.73      0.88      0.80      2081

    accuracy                           0.99     83325
   macro avg       0.90      0.95      0.92     83325
weighted avg       0.99      0.99      0.99     83325



In [19]:
check = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/data/section1/01_PB2/post_2020_logits_fold_0.npy')
ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, np.argmax(check, axis=1))

0.954472453842666 0.92419477991587 0.8982995914642359 0.954472453842666 0.9421877935572929
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     73749
           1       0.96      1.00      0.98      7495
           2       0.73      0.88      0.80      2081

    accuracy                           0.99     83325
   macro avg       0.90      0.95      0.92     83325
weighted avg       0.99      0.99      0.99     83325



In [20]:
np.testing.assert_allclose(logits, check, rtol=1e-3, atol=1e-2)

AssertionError: 
Not equal to tolerance rtol=0.001, atol=0.01

Mismatched elements: 19312 / 249975 (7.73%)
Max absolute difference: 0.0382483
Max relative difference: 38298.6
 x: array([[-0.574219,  2.375   , -1.015625],
       [-0.574219,  2.390625, -1.015625],
       [ 3.3125  , -1.875   , -0.388672],...
 y: array([[-0.576186,  2.376541, -1.015257],
       [-0.574948,  2.379976, -1.018405],
       [ 3.316324, -1.876641, -0.392945],...

In [21]:
logits[1000]

array([ 2.15625, -1.375  , -0.03125], dtype=float32)

In [22]:
check[1000]

array([ 2.1576655 , -1.375987  , -0.03716547], dtype=float32)

In [23]:
predict_prob[0, :]

NameError: name 'predict_prob' is not defined

In [24]:
check[0, :]

array([-0.5761862,  2.3765411, -1.0152571], dtype=float32)

In [25]:
for kfold_index, (train, test) in enumerate(splitter.split(X_train, y_train)):
    X_CV_train, y_CV_train = resampling(X_train[train], y_train[train], n_downsamples=16000, n_upsamples=600) # up sampling human and avian, keep non-human mammals (set 600)

    X_CV_test  = X_train[test]
    y_CV_test  = y_train[test]
    
    print ("Train shape: ", X_CV_train.shape, y_CV_train.shape)
    print (np.transpose(np.unique(y_CV_train, return_counts=True)))
    print ("Val shape: ", X_CV_test.shape, y_CV_test.shape)
    print (np.transpose(np.unique(y_CV_test, return_counts=True)))

    
    clf = WaveSeekerClassifier(
        n_channels=1,
        seq_L=seq_len,
        res_L=res_len,
        patch_size=patch_size,
        n_out=n_out,
        batch_size=batch_size,
        emb_dim=emb_dim,
        final_hidden_size=final_hidden_size,
        epochs=epochs,
        patch_mode="patch",
        wavelet_names=["sym4"],
        n_blocks=1,
        lr=0.0025)
    
    model_name = "Baseline"
    #clf.summary()

    clf.fit(X_CV_train, y_CV_train, X_CV_test, y_CV_test)#, save_path=model_weight)
    print("%s Result:" % model_name)
    print("High Quality (Post 2020)")
    ctest_high_quality = clf.predict(X_test_high_quality)
    
    ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, ctest_high_quality)
    param_results_high_quality.append((model_name, ba_main, f1_main, p_score_main, r_score_main, mcc_main))
    break 

Data Shape Before Sampling:  (70858, 5, 2400) (70858,)
Subtype/Host: 0 ,count:  42381 ,index count: (array([False,  True]), array([28477, 42381])) ,downsampling:  (16000, 5, 2400) (16000,)
Subtype/Host: 1 ,count:  20859 ,index count: (array([False,  True]), array([49999, 20859])) ,downsampling:  (16000, 5, 2400) (16000,)
Subtype/Host: 2 ,count:  7618 ,index count: (array([False,  True]), array([63240,  7618])) ,keep:  (7618, 5, 2400) (7618,)
Train shape:  (39618, 5, 2400) (39618,)
[[    0 16000]
 [    1 16000]
 [    2  7618]]
Val shape:  (7874, 5, 2400) (7874,)
[[   0 4709]
 [   1 2318]
 [   2  847]]
INFO | WaveSeekerNet.model | Using device: cuda:0
INFO | WaveSeekerNet.model | Trainable parameters: 1179123 / 1179123 total
INFO | WaveSeekerNet.model | Epoch 1/35 | BCE: 1.0586 | KAN: 0.0789 | SMoE: 0.1044 | Val Loss: 1.0197 | Val BA: 0.3333 | Train: 83.9s | Infer: 2.1s
INFO | WaveSeekerNet.model | Epoch 2/35 | BCE: 1.0214 | KAN: 0.0549 | SMoE: 0.0420 | Val Loss: 0.9419 | Val BA: 0.5057 

In [26]:
clf.summary()

AttributeError: 'WaveSeekerClassifier' object has no attribute 'summary'